# Mimi-VC: Voice Conversion via Neural Audio Codec Token Manipulation

This notebook implements **voice conversion** by exploiting the disentangled codebook structure of [Mimi](https://huggingface.co/kyutai/mimi), Kyutai's neural audio codec.

**Core idea**: Mimi's first codebook (CB0) captures **semantic/content** information (via WavLM distillation), while codebooks 1-7 capture **acoustic/speaker** details. By manipulating these codebooks, we can change the speaker identity while preserving speech content.

**Three approaches implemented:**
1. Naive codebook swap (baseline)
2. Retrieval-based VC (lookup table + FAISS)
3. Learned acoustic token translator (Transformer)

> **Runtime**: Select GPU runtime (Runtime > Change runtime type > T4 GPU)

## 1. Setup

In [ ]:
!pip install -q transformers torch torchaudio datasets faiss-cpu librosa matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np
import faiss
import matplotlib.pyplot as plt
from collections import defaultdict
from datasets import load_dataset
from transformers import MimiModel, AutoFeatureExtractor
from IPython.display import Audio as PlayAudio, display
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 2. Load Mimi Codec

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

mimi = MimiModel.from_pretrained('kyutai/mimi').to(device).eval()
feat_ext = AutoFeatureExtractor.from_pretrained('kyutai/mimi')

SR = feat_ext.sampling_rate          # 24000
N_CB = mimi.config.num_quantizers    # 8
CB_SIZE = mimi.config.codebook_size  # 2048
CB_DIM = mimi.config.codebook_dim    # 256
FRAME_RATE = mimi.config.frame_rate  # 12.5 Hz

print(f'Device: {device}')
print(f'Sample rate: {SR} Hz')
print(f'Codebooks: {N_CB} x {CB_SIZE} (dim={CB_DIM})')
print(f'Frame rate: {FRAME_RATE} Hz')
print(f'Codec structure:')
print(f'  CB0 = semantic (WavLM distilled) -> CONTENT')
print(f'  CB1-{N_CB-1} = acoustic residuals -> SPEAKER IDENTITY')

## 3. Helper Functions

In [ ]:
def encode(wav_np):
    """Encode numpy audio to Mimi tokens [1, N_CB, T]."""
    inputs = feat_ext(raw_audio=wav_np, sampling_rate=SR, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mimi.encode(inputs['input_values'], inputs.get('padding_mask'))
    return out.audio_codes.cpu()  # [1, N_CB, T]


def decode(codes):
    """Decode Mimi tokens to numpy audio."""
    with torch.no_grad():
        out = mimi.decode(codes.to(device))
    return out.audio_values.cpu().squeeze().numpy()


def play(wav_np, sr=SR, label=''):
    """Play audio in notebook."""
    if label:
        print(f'>> {label}')
    display(PlayAudio(wav_np, rate=sr))


def get_codebook_embeddings(codebook_idx=0):
    """Extract codebook embedding matrix [CB_SIZE, CB_DIM].
    Mimi uses MimiSplitResidualVectorQuantizer which splits into:
      - semantic_residual_vector_quantizer (CB0)
      - acoustic_residual_vector_quantizer (CB1-7)
    Each sub-quantizer has .layers[i].codebook.embed
    """
    q = mimi.quantizer
    # MimiSplitResidualVectorQuantizer path
    if hasattr(q, 'semantic_residual_vector_quantizer'):
        n_sem = q.num_semantic_quantizers
        if codebook_idx < n_sem:
            layer = q.semantic_residual_vector_quantizer.layers[codebook_idx]
        else:
            layer = q.acoustic_residual_vector_quantizer.layers[codebook_idx - n_sem]
    # Fallback: flat .layers
    elif hasattr(q, 'layers'):
        layer = q.layers[codebook_idx]
    else:
        raise RuntimeError(f'Unknown quantizer type: {type(q).__name__}')

    # Get embedding from the layer's codebook
    for attr in ['codebook.embed', 'codebook.embed_sum', 'codebook.embedding.weight']:
        obj = layer
        try:
            for part in attr.split('.'):
                obj = getattr(obj, part)
            w = obj.detach().cpu().float()
            if w.dim() == 3:
                w = w.squeeze(0)
            if w.shape[0] >= CB_SIZE:
                return w
        except AttributeError:
            continue
    raise RuntimeError('Cannot find codebook embeddings')


# Verify we can access codebook embeddings
cb0_emb = get_codebook_embeddings(0)
print(f'CB0 embedding shape: {cb0_emb.shape}')

## 4. Prepare Audio Data

We load audio from two different speakers. You can either:
- **Option A**: Use the built-in LibriSpeech demo dataset (default)
- **Option B**: Upload your own WAV files

In [ ]:
# ===== Option A: LibriSpeech demo dataset =====
ds = load_dataset(
    'hf-internal-testing/librispeech_asr_dummy', 'clean',
    split='validation', trust_remote_code=True
)

# Group samples by speaker
speaker_data = defaultdict(list)
for sample in ds:
    sid = sample['speaker_id']
    # Resample to Mimi's sample rate
    wav = torch.tensor(sample['audio']['array'], dtype=torch.float32)
    orig_sr = sample['audio']['sampling_rate']
    if orig_sr != SR:
        wav = torchaudio.functional.resample(wav, orig_sr, SR)
    speaker_data[sid].append(wav.numpy())

speaker_ids = sorted(speaker_data.keys())
print(f'Speakers found: {speaker_ids}')
print(f'Samples per speaker: {[(s, len(speaker_data[s])) for s in speaker_ids]}')

# Assign source and target
SRC_ID = speaker_ids[0]
TGT_ID = speaker_ids[1] if len(speaker_ids) > 1 else speaker_ids[0]

source_wav = speaker_data[SRC_ID][0]         # 1 utterance for conversion
target_wavs = speaker_data[TGT_ID]            # all utterances for index

print(f'\nSource speaker: {SRC_ID} ({len(source_wav)/SR:.2f}s)')
print(f'Target speaker: {TGT_ID} ({len(target_wavs)} utterances, '
      f'{sum(len(w) for w in target_wavs)/SR:.2f}s total)')

play(source_wav, label=f'Source speaker {SRC_ID}')
play(target_wavs[0], label=f'Target speaker {TGT_ID}')

In [ ]:
# ===== Option B: Upload your own files (uncomment to use) =====
# from google.colab import files
# import librosa
#
# print('Upload SOURCE audio (the voice to convert):')
# uploaded = files.upload()
# src_path = list(uploaded.keys())[0]
# source_wav, _ = librosa.load(src_path, sr=SR)
#
# print('Upload TARGET audio (the voice style to copy):')
# uploaded = files.upload()
# tgt_path = list(uploaded.keys())[0]
# tgt_wav, _ = librosa.load(tgt_path, sr=SR)
# target_wavs = [tgt_wav]

## 5. Encode & Analyze Codebook Structure

Visualize how Mimi distributes information across codebooks.
- **CB0** (semantic): should show similar patterns for same phonemes regardless of speaker
- **CB1-7** (acoustic): should differ between speakers for the same content

In [ ]:
# Encode audio
src_codes = encode(source_wav)  # [1, 8, T_src]
tgt_codes_list = [encode(w) for w in target_wavs]
tgt_codes_0 = tgt_codes_list[0]  # first target utterance for vis

print(f'Source tokens: {src_codes.shape} ({src_codes.shape[2]/FRAME_RATE:.2f}s)')
print(f'Target tokens: {tgt_codes_0.shape} ({tgt_codes_0.shape[2]/FRAME_RATE:.2f}s)')

# Visualize token heatmaps
fig, axes = plt.subplots(2, 1, figsize=(15, 6), constrained_layout=True)

im0 = axes[0].imshow(src_codes[0].numpy(), aspect='auto', cmap='viridis')
axes[0].set_ylabel('Codebook')
axes[0].set_title(f'Source Speaker {SRC_ID}')
axes[0].set_yticks(range(N_CB))
axes[0].set_yticklabels([f'CB{i} ({"sem" if i==0 else "aco"})' for i in range(N_CB)])
plt.colorbar(im0, ax=axes[0], shrink=0.6)

im1 = axes[1].imshow(tgt_codes_0[0].numpy(), aspect='auto', cmap='viridis')
axes[1].set_ylabel('Codebook')
axes[1].set_xlabel('Frame (12.5 Hz)')
axes[1].set_title(f'Target Speaker {TGT_ID}')
axes[1].set_yticks(range(N_CB))
axes[1].set_yticklabels([f'CB{i} ({"sem" if i==0 else "aco"})' for i in range(N_CB)])
plt.colorbar(im1, ax=axes[1], shrink=0.6)

plt.suptitle('Mimi Codec Token Visualization', fontsize=14)
plt.show()

# Token statistics
print('\nUnique tokens per codebook:')
print(f'{"CB":>4} {"Source":>8} {"Target":>8} {"Overlap":>8} {"Role":>10}')
for cb in range(N_CB):
    s_uniq = set(src_codes[0, cb].numpy().tolist())
    t_uniq = set(tgt_codes_0[0, cb].numpy().tolist())
    overlap = len(s_uniq & t_uniq)
    role = 'semantic' if cb == 0 else 'acoustic'
    print(f'{cb:>4} {len(s_uniq):>8} {len(t_uniq):>8} {overlap:>8} {role:>10}')

# Reconstruction test
print('\nReconstruction test (encode -> decode):')
recon_wav = decode(src_codes)
play(recon_wav, label='Reconstructed source (should sound identical to original)')

---
## Experiment 1: Naive Codebook Swap (Baseline)

Simply replace acoustic codebooks (CB1-7) from target speaker.
This is a **weak baseline** because RVQ residuals are computed relative to CB0 values - swapping them breaks the residual chain.

In [ ]:
min_len = min(src_codes.shape[2], tgt_codes_0.shape[2])

# Keep source CB0 (content), replace CB1-7 with target (speaker)
naive_codes = src_codes[:, :, :min_len].clone()
naive_codes[:, 1:, :] = tgt_codes_0[:, 1:, :min_len]

naive_wav = decode(naive_codes)

print('Naive swap: source CB0 + target CB1-7')
print('Expected: some voice change but likely artifacts from residual mismatch')
play(source_wav[:int(min_len/FRAME_RATE*SR)], label='Original source (truncated)')
play(naive_wav, label='Naive swap result')
play(target_wavs[0][:int(min_len/FRAME_RATE*SR)], label='Target reference (truncated)')

---
## Experiment 2: Retrieval-Based Voice Conversion

For each source frame, find the most similar semantic token in the target speaker's data, then use the **paired acoustic tokens** from that target frame.

This preserves residual consistency because semantic + acoustic tokens come from the same target frame.

**Two retrieval methods:**
- A) Direct lookup table on discrete CB0 token IDs
- B) FAISS on CB0 codebook embeddings (handles unseen tokens)

In [ ]:
# ===========================================================
# Build index from ALL target speaker utterances
# ===========================================================

# Collect all target frames: (semantic_token, acoustic_tokens)
tgt_sem_tokens = []    # list of int
tgt_acou_tokens = []   # list of np.array [7]
tgt_full_tokens = []   # list of np.array [8] (for decode)

for codes in tgt_codes_list:
    T = codes.shape[2]
    for t in range(T):
        tgt_sem_tokens.append(codes[0, 0, t].item())
        tgt_acou_tokens.append(codes[0, 1:, t].numpy())
        tgt_full_tokens.append(codes[0, :, t].numpy())

tgt_sem_arr = np.array(tgt_sem_tokens)
tgt_acou_arr = np.stack(tgt_acou_tokens)    # [N_frames, 7]
tgt_full_arr = np.stack(tgt_full_tokens)     # [N_frames, 8]

print(f'Target index: {len(tgt_sem_tokens)} frames from {len(tgt_codes_list)} utterances')
print(f'Unique semantic tokens in target: {len(set(tgt_sem_tokens))}')

# ----- Method A: Lookup table -----
lookup = defaultdict(list)
for i, sem in enumerate(tgt_sem_tokens):
    lookup[sem].append(i)

# Check coverage
src_sem = src_codes[0, 0].numpy()
covered = sum(1 for s in src_sem if s in lookup)
print(f'\nLookup coverage: {covered}/{len(src_sem)} '
      f'({100*covered/len(src_sem):.1f}%) source frames have exact match')

# ----- Method B: FAISS index on codebook embeddings -----
cb0_emb = get_codebook_embeddings(0).numpy()  # [2048, D]
D = cb0_emb.shape[1]

# Embed each target frame's semantic token
tgt_embeddings = cb0_emb[tgt_sem_arr].astype(np.float32)  # [N, D]
faiss.normalize_L2(tgt_embeddings)

# Build FAISS index
faiss_index = faiss.IndexFlatIP(D)  # cosine similarity
faiss_index.add(tgt_embeddings)
print(f'FAISS index: {faiss_index.ntotal} vectors, dim={D}')

In [ ]:
# ===========================================================
# Retrieval-based voice conversion
# ===========================================================

def retrieval_vc(src_codes, method='faiss', k=4, index_rate=1.0):
    """
    Convert source voice to target using retrieval.

    Args:
        src_codes: [1, N_CB, T] source Mimi tokens
        method: 'lookup' or 'faiss'
        k: number of neighbors for FAISS
        index_rate: blend ratio (1.0 = full target, 0.0 = keep source)
    Returns:
        converted_codes: [1, N_CB, T]
    """
    T = src_codes.shape[2]
    converted = src_codes.clone()

    if method == 'lookup':
        for t in range(T):
            sem = src_codes[0, 0, t].item()
            if sem in lookup:
                # Pick random matching frame from target
                idx = np.random.choice(lookup[sem])
                if index_rate >= 1.0:
                    converted[0, :, t] = torch.tensor(tgt_full_arr[idx])
                else:
                    # Keep source CB0, blend acoustic
                    converted[0, 1:, t] = torch.tensor(tgt_acou_arr[idx])

    elif method == 'faiss':
        # Embed all source semantic tokens
        src_sem = src_codes[0, 0].numpy()
        src_emb = cb0_emb[src_sem].astype(np.float32)  # [T, D]
        faiss.normalize_L2(src_emb)

        # Search k nearest neighbors
        distances, indices = faiss_index.search(src_emb, k)

        for t in range(T):
            # Weighted selection: closer neighbors are preferred
            # Use softmax of distances as weights
            weights = F.softmax(torch.tensor(distances[t]) * 10, dim=0).numpy()
            # Weighted random choice
            chosen = np.random.choice(k, p=weights)
            idx = indices[t, chosen]

            if index_rate >= 1.0:
                converted[0, :, t] = torch.tensor(tgt_full_arr[idx])
            else:
                converted[0, 1:, t] = torch.tensor(tgt_acou_arr[idx])

    return converted


# --- Run retrieval VC ---
print('Running retrieval-based VC...')

# Method A: Lookup (exact match)
np.random.seed(42)
lookup_codes = retrieval_vc(src_codes, method='lookup')
lookup_wav = decode(lookup_codes)

# Method B: FAISS (soft match)
np.random.seed(42)
faiss_codes = retrieval_vc(src_codes, method='faiss', k=4)
faiss_wav = decode(faiss_codes)

# Play results
play(source_wav, label='Original source')
play(lookup_wav, label='Retrieval VC (lookup - exact match)')
play(faiss_wav, label='Retrieval VC (FAISS - soft match)')
play(target_wavs[0], label='Target speaker reference')

In [ ]:
# ===========================================================
# Temporal smoothing for retrieval VC
# ===========================================================
# Without smoothing, each frame is independently matched, causing jitter.
# Apply simple sliding-window majority vote on retrieved tokens.

def retrieval_vc_smooth(src_codes, method='faiss', k=8, window=3):
    """
    Retrieval VC with temporal smoothing.
    For each frame, retrieve k candidates. Then in a sliding window,
    pick the candidate that appears most frequently.
    """
    T = src_codes.shape[2]

    # Get all candidates
    src_sem = src_codes[0, 0].numpy()
    src_emb = cb0_emb[src_sem].astype(np.float32)
    faiss.normalize_L2(src_emb)
    distances, indices = faiss_index.search(src_emb, k)

    # For each frame, pick the top-1 candidate
    selected = indices[:, 0].copy()

    # Sliding window smoothing: prefer candidates that are
    # temporally close to their neighbors in the target
    for t in range(1, T):
        prev_idx = selected[t - 1]
        # Among candidates for frame t, prefer one close to prev_idx
        candidates = indices[t]
        dists_to_prev = np.abs(candidates - prev_idx)
        # Combine semantic similarity with temporal proximity
        sem_scores = distances[t]
        temporal_scores = 1.0 / (1.0 + dists_to_prev.astype(np.float32))
        combined = sem_scores * 0.5 + temporal_scores * 0.5
        selected[t] = candidates[np.argmax(combined)]

    # Build converted codes
    converted = src_codes.clone()
    for t in range(T):
        converted[0, :, t] = torch.tensor(tgt_full_arr[selected[t]])

    return converted


smooth_codes = retrieval_vc_smooth(src_codes, k=8, window=3)
smooth_wav = decode(smooth_codes)

play(faiss_wav, label='FAISS retrieval (no smoothing)')
play(smooth_wav, label='FAISS retrieval (with temporal smoothing)')

---
## Experiment 3: Learned Acoustic Token Translator (Transformer)

Train a small Transformer that learns the mapping:
```
semantic_tokens (CB0) --> acoustic_tokens (CB1-7)
```
for a specific target speaker.

**Training**: Self-supervised on target speaker data only (no parallel data needed).
Given the target speaker's semantic tokens, predict their acoustic tokens.

**Inference**: Feed source speaker's semantic tokens through the trained model to generate target-style acoustic tokens.

In [ ]:
# ===========================================================
# Model Definition
# ===========================================================

class AcousticTranslator(nn.Module):
    """
    Predicts acoustic tokens (CB1-7) from semantic tokens (CB0)
    for a target speaker.
    """
    def __init__(
        self,
        n_semantic_tokens=2048,
        n_acoustic_tokens=2048,
        n_acoustic_codebooks=7,
        d_model=256,
        n_heads=4,
        n_layers=4,
        max_len=1024,
        dropout=0.1
    ):
        super().__init__()
        self.n_acoustic_codebooks = n_acoustic_codebooks

        # Semantic token embedding
        self.sem_embed = nn.Embedding(n_semantic_tokens, d_model)
        # Positional embedding
        self.pos_embed = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)

        # Transformer encoder (causal)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)

        # Output heads: one per acoustic codebook
        self.heads = nn.ModuleList([
            nn.Linear(d_model, n_acoustic_tokens)
            for _ in range(n_acoustic_codebooks)
        ])

    def forward(self, semantic_tokens):
        """
        Args:
            semantic_tokens: [B, T] int tensor of CB0 token indices
        Returns:
            logits: list of [B, T, V] tensors, one per acoustic codebook
        """
        B, T = semantic_tokens.shape
        x = self.sem_embed(semantic_tokens) + self.pos_embed[:, :T]

        # Causal mask
        mask = nn.Transformer.generate_square_subsequent_mask(T).to(x.device)
        x = self.transformer(x, mask=mask, is_causal=True)

        logits = [head(x) for head in self.heads]
        return logits


# Initialize model
translator = AcousticTranslator(
    n_semantic_tokens=CB_SIZE,
    n_acoustic_tokens=CB_SIZE,
    n_acoustic_codebooks=N_CB - 1,  # 7
    d_model=256,
    n_heads=4,
    n_layers=4
).to(device)

n_params = sum(p.numel() for p in translator.parameters())
print(f'AcousticTranslator: {n_params/1e6:.1f}M parameters')

In [ ]:
# ===========================================================
# Prepare Training Data
# ===========================================================
# Training data = target speaker's own (semantic, acoustic) token pairs
# No parallel data needed!

SEQ_LEN = 64  # training sequence length (64 frames = ~5 seconds)

# Concatenate all target speaker tokens
all_tgt_codes = torch.cat(tgt_codes_list, dim=2)  # [1, 8, T_total]
T_total = all_tgt_codes.shape[2]

train_sem = all_tgt_codes[0, 0]      # [T_total]
train_acou = all_tgt_codes[0, 1:]     # [7, T_total]

# Create training sequences
def make_batches(sem, acou, seq_len, batch_size=8):
    """Create random batches of training sequences."""
    T = sem.shape[0]
    n_seqs = max(1, T - seq_len)

    # Random starting positions
    starts = torch.randint(0, n_seqs, (batch_size,))

    sem_batch = torch.stack([sem[s:s+seq_len] for s in starts])
    acou_batch = torch.stack([acou[:, s:s+seq_len] for s in starts])

    return sem_batch.to(device), acou_batch.to(device)


print(f'Training data: {T_total} frames ({T_total/FRAME_RATE:.1f}s)')
print(f'Sequence length: {SEQ_LEN} frames ({SEQ_LEN/FRAME_RATE:.1f}s)')

# Quick test
test_sem, test_acou = make_batches(train_sem, train_acou, SEQ_LEN, 4)
print(f'Batch shapes: sem={test_sem.shape}, acou={test_acou.shape}')

In [ ]:
# ===========================================================
# Training Loop
# ===========================================================

N_STEPS = 500
BATCH_SIZE = 8
LR = 3e-4

optimizer = torch.optim.AdamW(translator.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, N_STEPS)

losses = []
translator.train()

for step in range(N_STEPS):
    sem_batch, acou_batch = make_batches(train_sem, train_acou, SEQ_LEN, BATCH_SIZE)

    # Forward: predict acoustic tokens from semantic tokens
    logits_list = translator(sem_batch)  # list of [B, T, V]

    # Loss: cross-entropy for each codebook
    loss = 0
    for cb_i, logits in enumerate(logits_list):
        target = acou_batch[:, cb_i, :]  # [B, T]
        loss += F.cross_entropy(logits.reshape(-1, CB_SIZE), target.reshape(-1))
    loss /= len(logits_list)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(translator.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    losses.append(loss.item())

    if (step + 1) % 100 == 0:
        print(f'Step {step+1}/{N_STEPS} | Loss: {loss.item():.4f} | '
              f'LR: {scheduler.get_last_lr()[0]:.2e}')

# Plot training loss
plt.figure(figsize=(10, 3))
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Acoustic Translator Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Final loss: {losses[-1]:.4f}')

In [ ]:
# ===========================================================
# Transformer VC Inference
# ===========================================================

@torch.no_grad()
def transformer_vc(src_codes, temperature=0.8, top_k=50):
    """
    Convert voice using trained acoustic translator.

    Args:
        src_codes: [1, N_CB, T] source tokens
        temperature: sampling temperature (lower = more deterministic)
        top_k: top-k sampling
    Returns:
        converted_codes: [1, N_CB, T]
    """
    translator.eval()

    sem_tokens = src_codes[0, 0:1].to(device)  # [1, T]

    # Predict acoustic tokens
    logits_list = translator(sem_tokens)  # list of [1, T, V]

    converted = src_codes.clone()
    for cb_i, logits in enumerate(logits_list):
        # Apply temperature
        logits = logits / temperature

        # Top-k filtering
        if top_k > 0:
            topk_vals, _ = torch.topk(logits, top_k, dim=-1)
            threshold = topk_vals[:, :, -1:]
            logits[logits < threshold] = float('-inf')

        # Sample
        probs = F.softmax(logits, dim=-1)
        predicted = torch.multinomial(
            probs.reshape(-1, CB_SIZE), 1
        ).reshape(1, -1)

        converted[0, cb_i + 1] = predicted.cpu()

    return converted


# Run inference with different temperatures
print('Transformer VC inference...')

torch.manual_seed(42)
trans_codes_08 = transformer_vc(src_codes, temperature=0.8)
trans_wav_08 = decode(trans_codes_08)

torch.manual_seed(42)
trans_codes_05 = transformer_vc(src_codes, temperature=0.5)
trans_wav_05 = decode(trans_codes_05)

torch.manual_seed(42)
trans_codes_02 = transformer_vc(src_codes, temperature=0.2)
trans_wav_02 = decode(trans_codes_02)

play(source_wav, label='Original source')
play(trans_wav_08, label='Transformer VC (temp=0.8, more diverse)')
play(trans_wav_05, label='Transformer VC (temp=0.5, balanced)')
play(trans_wav_02, label='Transformer VC (temp=0.2, more deterministic)')
play(target_wavs[0], label='Target speaker reference')

---
## Comparison & Analysis

In [ ]:
# ===========================================================
# Side-by-side comparison
# ===========================================================

print('=' * 60)
print('VOICE CONVERSION COMPARISON')
print('=' * 60)

results = {
    'Original source': source_wav,
    'Exp 1 - Naive swap': naive_wav,
    'Exp 2a - Lookup retrieval': lookup_wav,
    'Exp 2b - FAISS retrieval': faiss_wav,
    'Exp 2c - FAISS + smoothing': smooth_wav,
    'Exp 3 - Transformer (t=0.5)': trans_wav_05,
    'Target reference': target_wavs[0],
}

for label, wav in results.items():
    play(wav, label=label)
    print()

In [ ]:
# ===========================================================
# Visualize converted token patterns
# ===========================================================

fig, axes = plt.subplots(4, 1, figsize=(15, 12), constrained_layout=True)

display_len = min(src_codes.shape[2], 200)  # first 200 frames

titles = [
    f'Source Speaker {SRC_ID} (original)',
    'Retrieval VC (FAISS + smoothing)',
    'Transformer VC (temp=0.5)',
    f'Target Speaker {TGT_ID} (reference)',
]
all_codes = [
    src_codes[0, :, :display_len],
    smooth_codes[0, :, :display_len],
    trans_codes_05[0, :, :display_len],
    tgt_codes_0[0, :, :display_len],
]

for ax, title, codes in zip(axes, titles, all_codes):
    im = ax.imshow(codes.numpy(), aspect='auto', cmap='viridis')
    ax.set_ylabel('Codebook')
    ax.set_title(title)
    ax.set_yticks(range(N_CB))
    ax.set_yticklabels([f'CB{i}' for i in range(N_CB)])
    plt.colorbar(im, ax=ax, shrink=0.6)

axes[-1].set_xlabel('Frame')
plt.suptitle('Token Comparison Across Methods', fontsize=14)
plt.show()

---
## Ablation: Which Codebooks Matter?

Test which codebooks carry speaker identity vs content by selectively swapping subsets.

In [ ]:
# ===========================================================
# Codebook ablation study
# ===========================================================
# For each experiment, keep some codebooks from source,
# replace others from target (using retrieval)

ablation_configs = [
    ('All source (original)', list(range(8)), []),
    ('Swap CB0 only (semantic)', [], [0]),
    ('Swap CB1 only', list(range(1)), [1]),
    ('Swap CB1-3', [0], [1, 2, 3]),
    ('Swap CB1-7 (all acoustic)', [0], [1, 2, 3, 4, 5, 6, 7]),
    ('Swap ALL (full target)', [], [0, 1, 2, 3, 4, 5, 6, 7]),
]

print('Codebook Ablation Study')
print('Which codebooks carry speaker identity?\n')

for label, keep_src, swap_tgt in ablation_configs:
    T = min(src_codes.shape[2], smooth_codes.shape[2])
    abl_codes = src_codes[:, :, :T].clone()

    for cb in swap_tgt:
        abl_codes[0, cb, :] = smooth_codes[0, cb, :T]

    abl_wav = decode(abl_codes)
    play(abl_wav, label=label)

---
## Summary

| Method | Quality | Speed | Training Required |
|--------|---------|-------|-------------------|
| Naive swap | Low (artifacts) | Instant | No |
| Lookup retrieval | Medium | Instant | No |
| FAISS retrieval | Medium-High | Fast | No |
| FAISS + smoothing | High | Fast | No |
| Transformer | High | Fast | Yes (minutes) |

### Key findings:
1. **CB0 = content**: Mimi's first codebook successfully encodes speaker-independent semantic information
2. **CB1-7 = speaker identity**: Replacing acoustic codebooks changes perceived voice
3. **Retrieval preserves residual coherence**: Using paired (semantic, acoustic) tokens from the same target frame avoids RVQ residual mismatch
4. **Temporal smoothing is important**: Independent frame-level retrieval causes jitter
5. **Transformer learns speaker style**: Given enough data, the model captures the target speaker's acoustic patterns

### Next steps:
- Train on more target speaker data (30+ minutes)
- Add F0 (pitch) normalization for cross-gender conversion
- Implement streaming inference for real-time VC
- Add speaker embedding conditioning for multi-speaker support
- Try diffusion in continuous latent space before RVQ (Approach C from design doc)